# 🧠 CalRetail — AI Chatbot (Nexa Support)
## Goal
Generate natural, context-aware customer support answers grounded in the customer's real
orders and returns.

## Algorithmic Explanation
**LangChain LLM generation, grounded in real account context, with a rule-based safety net**
1. Fetch the customer's real recent orders and returns (name, order status/dates, return
   reasons/status) — never fabricated.
2. When an LLM provider is configured, send that real context to the LLM (via
   `backend.utils.llm_service`) with instructions to answer *only* from the given context and
   to flag a human escalation when appropriate.
3. If no LLM is configured, or its response can't be parsed, fall back to a deterministic
   rule-based responder built on the same real order/return context.



In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data' / 'processed'
if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")

In [ ]:
orders = pd.read_csv(processed_dir / 'orders.csv')
returns = pd.read_csv(processed_dir / 'returns.csv')
cust = pd.read_csv(processed_dir / 'customers.csv')

print(f"Chatbot support models loaded. Orders: {len(orders)} | Returns: {len(returns)}")


In [ ]:
from backend.utils.llm_service import llm_chat, llm_available

CHATBOT_SYSTEM_TEMPLATE = """You are "Nexa", CalRetail's friendly customer support chatbot.
Answer the customer's question using ONLY the real account context below — never invent order
IDs, dates, statuses or amounts that aren't given to you. If the context doesn't contain the
answer, politely say you don't have that information and offer to escalate to a human agent.

Customer ID: {customer_id}
Customer name: {name}
Recent orders: {orders_context}
Recent returns: {returns_context}

Return ONLY a JSON object with this exact shape:
{{
  "response": "<your natural-language reply to the customer, 1-3 sentences>",
  "intent": "order_status" | "return_status" | "complaint" | "general",
  "escalate": true | false
}}"""


def _rule_based_response(customer_id, question):
    """Deterministic fallback used when no LLM is configured, or the LLM
    output can't be parsed. Also used directly by the Mock LLM stand-in so
    the flow is fully exercised even without a provider API key."""
    cust_orders = orders[orders['customer_id'] == customer_id].sort_values('order_date', ascending=False).head(3)
    cust_returns = returns[returns['customer_id'] == customer_id].sort_values('return_date', ascending=False).head(2)
    cname = cust[cust['customer_id'] == customer_id].iloc[0]['name'].split()[0] if customer_id in cust['customer_id'].values else "there"

    q_lower = question.lower()
    if 'order' in q_lower or 'delivery' in q_lower:
        if not cust_orders.empty:
            o = cust_orders.iloc[0]
            answer = f"Hi {cname}, your recent order {o['order_id']} is currently '{o['status']}'. It was ordered on {o['order_date']}."
        else:
            answer = f"Hi {cname}, I couldn't find any recent orders associated with your profile."
        intent = "order_status"
    elif 'return' in q_lower or 'refund' in q_lower:
        if not cust_returns.empty:
            r = cust_returns.iloc[0]
            answer = f"Hi {cname}, return request {r['return_id']} was received: status is {r['status'] or 'Processing'}."
        else:
            answer = f"Hi {cname}, you have no active return requests on file."
        intent = "return_status"
    else:
        answer = f"Hi {cname}! Welcome to CalRetail Support. How can I assist you with orders or returns today?"
        intent = "general"

    escalate = "complaint" in q_lower or "defect" in q_lower
    return {
        "response": answer,
        "intent": "complaint" if escalate else intent,
        "escalate": escalate,
    }


def chatbot_response(customer_id, question):
    cust_orders = orders[orders['customer_id'] == customer_id].sort_values('order_date', ascending=False).head(3)
    cust_returns = returns[returns['customer_id'] == customer_id].sort_values('return_date', ascending=False).head(2)
    cname = cust[cust['customer_id'] == customer_id].iloc[0]['name'].split()[0] if customer_id in cust['customer_id'].values else "there"

    orders_context = "; ".join(
        f"Order {o['order_id']}: {o['status']}, placed {o['order_date']}, total Rs.{o['total_amount']}"
        for _, o in cust_orders.iterrows()
    ) or "No recent orders on file."
    returns_context = "; ".join(
        f"Return {r['return_id']}: {r['status']}, reason {r['reason']}"
        for _, r in cust_returns.iterrows()
    ) or "No return requests on file."

    used_llm = False
    result = None
    if llm_available():
        prompt = CHATBOT_SYSTEM_TEMPLATE.format(
            customer_id=customer_id, name=cname,
            orders_context=orders_context, returns_context=returns_context
        )
        raw = llm_chat(messages=[("system", prompt), ("human", question)], fallback="")
        if raw:
            try:
                cleaned = raw.strip()
                if cleaned.startswith("```"):
                    cleaned = cleaned.strip("`")
                    cleaned = cleaned[4:] if cleaned.lower().startswith("json") else cleaned
                parsed = json.loads(cleaned.strip())
                if "response" in parsed:
                    result = parsed
                    used_llm = True
            except Exception:
                result = None

    if result is None:
        result = _rule_based_response(customer_id, question)

    return {
        "customer_id": customer_id,
        "query": question,
        "response": result.get("response", ""),
        "intent": result.get("intent", "general"),
        "escalate": bool(result.get("escalate", False)),
        "powered_by": "LangChain LLM" if used_llm else "Rule-Based Engine",
    }

backend_res = chatbot_response("C0001", "Where is my order?")
print("AI Chatbot payload:\n", json.dumps(backend_res, indent=2))

In [ ]:
print("=== CALRETAIL CHATBOT INTERFACE ===")
print(f"Customer Question: 'Where is my order?'")
print(f"Nexa Chatbot: {backend_res['response']}")
print(f"Needs human escalation? {'YES' if backend_res['escalate'] else 'NO'}")
